# Agente Text-to-SQL — CineData Analytics

Este notebook implementa um agente Text-to-SQL utilizando PydanticAI, OpenRouter e SQLite.

O fluxo foi inspirado de forma simplificada na arquitetura CHESS, separando etapas de schema linking, geração de SQL, execução, validação e correção de consultas.

## 1. Configuração do ambiente

Nesta seção são carregadas as dependências, variáveis de ambiente e configurações gerais utilizadas pelo agente.

In [49]:
import os
import re
import sqlite3
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent, PromptedOutput

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY não encontrada no arquivo .env")

In [28]:
# Carregando constantes do projeto
MODEL_NAME = "z-ai/glm-5.2:free"
DATABASE_PATH = Path("cinerocket.db")
DB_PATH = str(DATABASE_PATH)

In [3]:
# Validando se o banco está disponível localmente
if not DATABASE_PATH.exists():
    raise FileNotFoundError(
        f"Banco de dados não encontrado em: {DATABASE_PATH.resolve()}"
    )


### 1.2 Configuração do modelo

O modelo é acessado através da API da OpenRouter, utilizando uma interface compatível com a API da OpenAI.

In [16]:
pydantic_model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    ),
)

In [ ]:
# Testando modelo
test_agent = Agent(
    pydantic_model,
    instructions="Responda apenas com OK",
)

result = await test_agent.run("Teste de conexão")
print(result.output)

OK


## 2. Utilitários do banco de dados

Funções auxiliares responsáveis pela conexão e inspeção do banco SQLite.

Essas funções são mantidas separadas da lógica dos agentes para facilitar testes, reutilização e manutenção.

In [4]:
# Funçao de conexão com banco
def get_connection(database_path: Path = DATABASE_PATH) -> sqlite3.Connection:
    """
    Cria uma conexão com o banco SQLite utilizado pelo agente.

    Args:
        database_path: Caminho para o arquivo SQLite.

    Returns:
        Uma conexão ativa com o banco de dados SQLite.

    Raises:
        FileNotFoundError: Caso o arquivo do banco não exista.
    """
    if not database_path.exists():
        raise FileNotFoundError(
            f"Banco de dados não encontrado: {database_path}"
        )

    return sqlite3.connect(database_path)

In [5]:
# Pegando os nomes das tabelas do banco
def get_table_names(conn: sqlite3.Connection) -> list[str]:
    """
    Retorna os nomes das tabelas disponíveis no banco SQLite.

    Args:
        conn: Conexão ativa com o banco SQLite.

    Returns:
        Lista contendo os nomes das tabelas encontradas.
    """
    query = """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
    """

    cursor = conn.execute(query)

    return [row[0] for row in cursor.fetchall()]

In [6]:
# Verificando se as tabelas foram carregadas corretamente
with get_connection() as conn:
    tables = get_table_names(conn)

tables

['alembic_version',
 'bridge_movie_company',
 'bridge_movie_genre',
 'bridge_movie_person',
 'dim_companies',
 'dim_genres',
 'dim_movies',
 'dim_people',
 'dim_reviews',
 'fact_movies_performance',
 'movie_reviews']

### 2.1 Inspeção do esquema das tabelas

In [7]:
def get_table_schema(
    conn: sqlite3.Connection,
    table_name: str,
) -> list[dict[str, Any]]:
    """
    Retorna informações sobre as colunas de uma tabela SQLite.

    Args:
        conn: Conexão ativa com o banco SQLite.
        table_name: Nome da tabela que será inspecionada.

    Returns:
        Lista de dicionários contendo informações sobre as colunas.

    Raises:
        ValueError: Caso a tabela informada não exista no banco.
    """
    available_tables = get_table_names(conn)

    if table_name not in available_tables:
        raise ValueError(
            f"Tabela '{table_name}' não encontrada no banco."
        )

    cursor = conn.execute(f"PRAGMA table_info({table_name});")

    columns = []

    for row in cursor.fetchall():
        columns.append(
            {
                "name": row[1],
                "type": row[2],
                "not_null": bool(row[3]),
                "primary_key": bool(row[5]),
            }
        )

    return columns


def get_database_schema(
    db_path: str,
    ignored_tables: set[str] | None = None,
) -> str:
    """
    Retorna uma representação textual compacta do esquema do banco SQLite.

    A função lista tabelas e respectivas colunas para uso no processo
    de schema linking. Tabelas técnicas podem ser ignoradas.

    Args:
        db_path: Caminho para o arquivo SQLite.
        ignored_tables: Conjunto opcional de tabelas que não devem
            aparecer no esquema.

    Returns:
        Representação textual do esquema do banco.

    Raises:
        FileNotFoundError: Caso o arquivo do banco não exista.
    """
    if ignored_tables is None:
        ignored_tables = {"alembic_version"}

    if not Path(db_path).exists():
        raise FileNotFoundError(f"Banco não encontrado: {db_path}")

    conn = sqlite3.connect(db_path)

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name;
            """
        )

        tables = [
            row[0]
            for row in cursor.fetchall()
            if row[0] not in ignored_tables
        ]

        schema_parts: list[str] = []

        for table in tables:
            cursor.execute(f'PRAGMA table_info("{table}")')

            columns = cursor.fetchall()

            schema_parts.append(f"Tabela: {table}")

            for column in columns:
                name = column[1]
                data_type = column[2]
                not_null = bool(column[3])
                primary_key = bool(column[5])

                metadata: list[str] = []

                if primary_key:
                    metadata.append("PK")

                if not_null:
                    metadata.append("NOT NULL")

                suffix = (
                    f" [{' | '.join(metadata)}]"
                    if metadata
                    else ""
                )

                schema_parts.append(
                    f"  - {name}: {data_type}{suffix}"
                )

            schema_parts.append("")

        return "\n".join(schema_parts)

    finally:
        conn.close()

> **Observação:** o esquema abaixo é utilizado apenas na etapa de *schema linking*.  
> O gerador de SQL receberá posteriormente apenas o subconjunto do esquema considerado relevante para cada pergunta.

In [8]:
schema = get_database_schema(DB_PATH)
print(schema)

Tabela: bridge_movie_company
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - sk_company_id: VARCHAR(64) [PK | NOT NULL]

Tabela: bridge_movie_genre
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - sk_genre_id: VARCHAR(64) [PK | NOT NULL]

Tabela: bridge_movie_person
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - sk_person_id: VARCHAR(64) [PK | NOT NULL]

Tabela: dim_companies
  - sk_company_id: VARCHAR(64) [PK | NOT NULL]
  - nome_produtora: VARCHAR(255) [NOT NULL]

Tabela: dim_genres
  - sk_genre_id: VARCHAR(64) [PK | NOT NULL]
  - nome_genero: VARCHAR(50) [NOT NULL]

Tabela: dim_movies
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - id_filme: VARCHAR(50) [NOT NULL]
  - titulo: VARCHAR(500) [NOT NULL]
  - data_lancamento: DATE
  - ano_lancamento: INTEGER
  - duracao_minutos: INTEGER
  - idioma_original: VARCHAR(10)
  - status_filme: VARCHAR(50)
  - sinopse: VARCHAR(4000)
  - url_poster: VARCHAR(2048)
  - url_backdrop: VARCHAR(2048)

Tabela: dim_people
  - sk_person_id: VARCHAR(64)

## 3. Representação dos relacionamentos

Como o banco utiliza tabelas dimensão, fato e tabelas de associação, os principais relacionamentos são explicitados para auxiliar o processo de schema linking.

In [ ]:
# Os relacionamentos são utilizados como contexto estrutural e não representam dados produzidos pelo modelo
RELATIONSHIPS = [
    "bridge_movie_company.sk_movie_id = dim_movies.sk_movie_id",
    "bridge_movie_company.sk_company_id = dim_companies.sk_company_id",

    "bridge_movie_genre.sk_movie_id = dim_movies.sk_movie_id",
    "bridge_movie_genre.sk_genre_id = dim_genres.sk_genre_id",

    "bridge_movie_person.sk_movie_id = dim_movies.sk_movie_id",
    "bridge_movie_person.sk_person_id = dim_people.sk_person_id",

    "fact_movies_performance.sk_movie_id = dim_movies.sk_movie_id",

    "dim_reviews.sk_movie_id = dim_movies.sk_movie_id",

    "movie_reviews.sk_movie_id = dim_movies.sk_movie_id",
]

## 4. Schema Linking

O schema linking identifica o conjunto mínimo de tabelas, colunas, filtros e relacionamentos necessários para responder a uma pergunta em linguagem natural.

Essa etapa reduz o contexto fornecido posteriormente ao gerador de SQL.

In [50]:
# Modelo de saída do schema linking
class SchemaLink(BaseModel):
    """Elementos mínimos do esquema necessários para responder à pergunta."""

    tables: list[str] = Field(
        description="Nomes das tabelas necessárias para responder à pergunta."
    )

    columns: list[str] = Field(
        description=(
            "Colunas necessárias no formato 'tabela.coluna'. "
            "Exemplo: 'dim_movies.titulo'."
        )
    )

    filters: list[str] = Field(
        default_factory=list,
        description=(
            "Condições de filtro necessárias para responder à pergunta. "
            "Não invente valores que não estejam presentes na pergunta."
        )
    )

    joins: list[str] = Field(
    default_factory=list,
    description=(
        "Relacionamentos necessários entre as tabelas. "
        "Use referências 'tabela.coluna' quando houver tabelas distintas. "
        "Quando a mesma tabela precisar participar mais de uma vez, "
        "descreva semanticamente a auto-junção sem criar aliases SQL."
        )
    )

    reasoning: str = Field(
        description=(
            "Explicação breve de por que os elementos foram selecionados."
        )
    )
    

# Constrói o contexto global do schema linker
def get_schema_for_linking(db_path: str) -> str:
    """
    Constrói o contexto global utilizado pelo agente de schema linking.

    Inclui a estrutura das tabelas analíticas e os relacionamentos
    conhecidos entre elas.

    Args:
        db_path: Caminho para o banco SQLite.

    Returns:
        Contexto textual contendo esquema e relacionamentos.
    """
    schema = get_database_schema(db_path)

    relationships_text = "\n".join(
        f"- {relationship}"
        for relationship in RELATIONSHIPS
    )

    return f"""
ESQUEMA:

{schema}

RELACIONAMENTOS:

{relationships_text}
""".strip()

### 4.1 Agente de schema linking

In [45]:
schema_link_agent = Agent(
    pydantic_model,
    output_type=PromptedOutput(SchemaLink),
    instructions=f"""
Você é responsável pelo schema linking de um sistema Text-to-SQL.

Sua única tarefa é identificar o MENOR conjunto de elementos do banco
necessário para responder à pergunta do usuário.

Use exclusivamente o esquema fornecido.

Regras:

1. Nunca invente tabelas ou colunas.
2. Retorne apenas tabelas existentes no esquema.
3. Toda coluna deve estar no formato "tabela.coluna".
4. Se uma pergunta exigir mais de uma tabela, identifique os joins necessários.
5. Em filters, descreva apenas filtros necessários para responder à pergunta.
6. Não invente valores de filtro que não estejam explicitamente presentes na pergunta.
7. Não gere SQL.
8. Não responda à pergunta do usuário.
9. Se uma tabela bridge for necessária para conectar entidades, inclua-a.
10. Utilize apenas os relacionamentos fornecidos.
11. Prefira sempre o menor conjunto possível de tabelas e colunas.
12. Preencha todos os campos da saída estruturada.

13. Não inclua tabelas intermediárias que não sejam necessárias para conectar
    os elementos selecionados.

14. Quando um filtro depender de valores categóricos do banco que não estejam
    explicitamente escritos na pergunta, NÃO escreva valores literais.

    Exemplo incorreto:
    "dim_people.tipo_pessoa = 'ator'"

    Exemplo correto:
    "filtrar dim_people.tipo_pessoa para representar pessoas com papel de ator"

    Os valores reais serão consultados diretamente no banco em outra etapa.

15. Uma mesma tabela pode ser necessária em múltiplos papéis lógicos.
    Nesse caso, inclua a tabela apenas uma vez em tables.

16. Não inclua uma tabela apenas por integridade referencial ou contexto conceitual.
    Inclua somente tabelas necessárias para obter colunas, aplicar filtros ou
    estabelecer os relacionamentos necessários à resposta.

    Se o resultado puder ser obtido sem uma tabela, ela deve ser omitida.

17. Não crie aliases SQL no schema linking. Descreva apenas as relações
    estruturais necessárias. A criação de aliases será responsabilidade
    do gerador de SQL.

18. Em reasoning, explique apenas por que os elementos do esquema foram
    selecionados. Não descreva a implementação completa da consulta SQL.

{get_schema_for_linking(DB_PATH)}
""",
)

### 4.2 Teste do agente de schema linking

In [43]:
question = "Top 10 filmes com maior receita em R$"

result = await schema_link_agent.run(question)

linked = result.output

print("Tables:   ", linked.tables)
print("Columns:  ", linked.columns)
print("Filters:  ", linked.filters)
print("Joins:    ", linked.joins)
print("Reasoning:", linked.reasoning)

Tables:    ['dim_movies', 'fact_movies_performance']
Columns:   ['dim_movies.sk_movie_id', 'dim_movies.titulo', 'fact_movies_performance.sk_movie_id', 'fact_movies_performance.receita_brl']
Filters:   []
Joins:     ['fact_movies_performance.sk_movie_id = dim_movies.sk_movie_id']
Reasoning: Para obter os 10 filmes com maior receita em reais (R$), preciso do título do filme na dim_movies e da receita em BRL na fact_movies_performance. A junção é feita via sk_movie_id. Não há filtros adicionais além da ordenação decrescente por receita_brl e limite de 10.


In [46]:
question = "Qual dupla ator-diretor mais trabalhou junta?"

result = await schema_link_agent.run(question)
linked = result.output

print("Tables:   ", linked.tables)
print("Columns:  ", linked.columns)
print("Filters:  ", linked.filters)
print("Joins:    ", linked.joins)
print("Reasoning:", linked.reasoning)

Tables:    ['dim_movies', 'dim_people', 'bridge_movie_person']
Columns:   ['dim_movies.sk_movie_id', 'dim_people.sk_person_id', 'dim_people.nome_pessoa', 'dim_people.tipo_pessoa', 'bridge_movie_person.sk_movie_id', 'bridge_movie_person.sk_person_id']
Filters:   ['filtrar dim_people.tipo_pessoa para representar pessoas com papel de ator', 'filtrar dim_people.tipo_pessoa para representar pessoas com papel de diretor']
Joins:     ['bridge_movie_person.sk_movie_id = dim_movies.sk_movie_id', 'bridge_movie_person.sk_person_id = dim_people.sk_person_id']
Reasoning: Para encontrar a dupla ator-diretor que mais trabalhou junta, precisamos: (1) dim_movies para identificar filmes; (2) dim_people para obter nomes e distinguir atores de diretores via tipo_pessoa; (3) bridge_movie_person para relacionar filmes às pessoas. A consulta exigirá unir bridge_movie_person duas vezes (uma para atores, outra para diretores) e contar co-ocorrências por par ator-diretor.


## 5. Construção do focused_context

Após o schema linking, apenas as tabelas e colunas selecionadas são utilizadas para construir o contexto enviado ao gerador de SQL.

Isso reduz informações irrelevantes no prompt e ajuda a diminuir a geração de tabelas ou colunas inexistentes.

In [51]:
# Seleciona colunas relevantes
def get_columns_for_table(
    linked: SchemaLink,
    table: str,
) -> list[str]:
    """
    Retorna as colunas relevantes de uma tabela.

    Inclui tanto as colunas selecionadas explicitamente pelo schema linker
    quanto as colunas necessárias para realizar os joins.

    Args:
        linked: Resultado estruturado do schema linking.
        table: Nome da tabela.

    Returns:
        Lista ordenada com os nomes das colunas relevantes.
    """
    prefix = f"{table}."

    selected_columns = {
        column.removeprefix(prefix)
        for column in linked.columns
        if column.startswith(prefix)
    }

    join_columns = set(
        get_join_columns(linked, table)
    )

    return sorted(
        selected_columns | join_columns
    )


# Auxilia get_columns_for_table extraindo as colunas nos joins
def get_join_columns(
    linked: SchemaLink,
    table: str,
) -> list[str]:
    """
    Extrai as colunas de uma tabela mencionadas nos joins identificados
    pelo schema linker.

    Args:
        linked: Resultado estruturado do schema linking.
        table: Nome da tabela.

    Returns:
        Lista de nomes de colunas da tabela utilizadas em joins.
    """
    pattern = rf"\b{re.escape(table)}\.([A-Za-z_][A-Za-z0-9_]*)\b"

    columns: set[str] = set()

    for join in linked.joins:
        columns.update(re.findall(pattern, join))

    return sorted(columns) 
 
# Excluindo chaves surrogate sem removê-las da definição de colunas
def get_sample_columns(columns: list[str]) -> list[str]:
    """
    Remove das amostras colunas técnicas de chave surrogate.

    As colunas continuam presentes no contexto estrutural, mas seus valores
    não são enviados ao modelo para evitar consumo desnecessário de tokens.

    Args:
        columns: Lista de colunas relevantes.

    Returns:
        Lista de colunas apropriadas para amostragem.
    """
    return [
        column
        for column in columns
        if not column.startswith("sk_")
    ]

   
# Pega uma amostra de linhas   
def get_sample_rows(
    db_path: str,
    table: str,
    columns: list[str],
    limit: int = 3,
) -> list[dict[str, object]]:
    """
    Obtém uma pequena amostra de linhas das colunas relevantes.

    Args:
        db_path: Caminho para o banco SQLite.
        table: Nome da tabela consultada.
        columns: Colunas que devem aparecer na amostra.
        limit: Número máximo de linhas retornadas.

    Returns:
        Lista de dicionários representando as linhas consultadas.
    """
    
    if not columns:
        return []

    quoted_columns = ", ".join(
        f'"{column}"'
        for column in columns
    )

    query = f"""
        SELECT {quoted_columns}
        FROM "{table}"
        LIMIT ?
    """

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    try:
        cursor = conn.execute(query, (limit,))

        return [
            dict(row)
            for row in cursor.fetchall()
        ]

    finally:
        conn.close()
        
# Valores categóricos relevantes
VALUE_SAMPLE_COLUMNS = {
    "dim_people.tipo_pessoa",
    "dim_movies.status_filme",
    "dim_movies.idioma_original",
}

       
def get_column_values(
    db_path: str,
    table: str,
    column: str,
    limit: int = 10,
) -> list[object]:
    """
    Retorna valores distintos não nulos de uma coluna.

    Args:
        db_path: Caminho para o banco SQLite.
        table: Nome da tabela.
        column: Nome da coluna.
        limit: Quantidade máxima de valores retornados.

    Returns:
        Lista de valores distintos encontrados na coluna.
    """
    
    if limit <= 0:
        return []
    
    query = f"""
        SELECT DISTINCT "{column}"
        FROM "{table}"
        WHERE "{column}" IS NOT NULL
        LIMIT ?
    """

    conn = sqlite3.connect(db_path)

    try:
        cursor = conn.execute(query, (limit,))
        return [row[0] for row in cursor.fetchall()]

    finally:
        conn.close()
        
# Montando o focused_context  
def build_focused_context(
    linked: SchemaLink,
    db_path: str,
) -> str:
    """
    Constrói o contexto mínimo utilizado pelo gerador SQL.

    O contexto inclui:
    - tabelas selecionadas pelo schema linker;
    - colunas relevantes;
    - pequenas amostras dessas colunas;
    - joins necessários;
    - filtros semânticos;
    - valores categóricos observados no banco.

    Args:
        linked: Resultado estruturado do schema linking.
        db_path: Caminho para o banco SQLite.

    Returns:
        Contexto textual focado para geração da consulta SQL.
    """
    parts: list[str] = []

    for table in linked.tables:
        relevant_columns = get_columns_for_table(
            linked,
            table,
        )

        if not relevant_columns:
            continue

        parts.append(
            f"### TABELA: {table}"
        )

        columns_text = "\n".join(
            f"- {column}"
            for column in relevant_columns
        )

        parts.append(
            f"Colunas relevantes:\n{columns_text}"
        )

        samples_columns = get_sample_columns(
            relevant_columns
        )

        samples = get_sample_rows(
            db_path,
            table,
            samples_columns,
        )

        if samples:
            parts.append(
                f"Amostras relevantes: {samples}"
            )

    if linked.joins:
        parts.append(
            "### JOINS NECESSÁRIOS"
        )

        parts.extend(
            f"- {join}"
            for join in linked.joins
        )

    if linked.filters:
        parts.append(
            "### FILTROS IDENTIFICADOS"
        )

        parts.extend(
            f"- {filter_condition}"
            for filter_condition in linked.filters
        )

    values_section: list[str] = []

    for column_ref in linked.columns:
        if column_ref not in VALUE_SAMPLE_COLUMNS:
            continue

        table, column = column_ref.split(".", 1)

        values = get_column_values(
            db_path,
            table,
            column,
        )

        if values:
            values_section.append(
                f"{column_ref}: {values}"
            )

    if values_section:
        parts.append(
            "### VALORES CATEGÓRICOS OBSERVADOS"
        )

        parts.extend(values_section)

    return "\n\n".join(parts)    


### 5.1 Teste do focused_context

In [52]:
question = "Qual dupla ator-diretor mais trabalhou junta?"

result = await schema_link_agent.run(question)
linked = result.output

context = build_focused_context(
    linked,
    DB_PATH,
)

print(context)

### TABELA: dim_movies

Colunas relevantes:
- sk_movie_id
- titulo

Amostras relevantes: [{'titulo': '"""blessed"""'}, {'titulo': '"""truelove: The Film"""'}, {'titulo': '"baba ""bu Alem Bi Alem"""'}]

### TABELA: dim_people

Colunas relevantes:
- nome_pessoa
- sk_person_id
- tipo_pessoa

Amostras relevantes: [{'nome_pessoa': '"" The Ripping Movie Of The Summer In Theaters!"', 'tipo_pessoa': 'Roteirista'}, {'nome_pessoa': '"""you Win Some. You Lose Most."""', 'tipo_pessoa': 'Diretor'}, {'nome_pessoa': '""loro"" Wishes To Tell The Story Of Some Italians', 'tipo_pessoa': 'Roteirista'}]

### TABELA: bridge_movie_person

Colunas relevantes:
- sk_movie_id
- sk_person_id

### JOINS NECESSÁRIOS

- bridge_movie_person.sk_movie_id = dim_movies.sk_movie_id

- bridge_movie_person.sk_person_id = dim_people.sk_person_id

### FILTROS IDENTIFICADOS

- filtrar dim_people.tipo_pessoa para representar pessoas com papel de ator

- filtrar dim_people.tipo_pessoa para representar pessoas com papel de diret

## 6. Geração da consulta SQL

Nesta etapa, um segundo agente recebe apenas o contexto selecionado pelo schema linking e gera uma consulta SQL para responder à pergunta do usuário.